# Economic indicators

World Bank panel since 2000. The pipeline is **`pipeline.py`**: facts packet → OLS + Ridge baseline → optional LLM briefing → **`report.html`**.

This notebook loads the cache, shows the packet, and rebuilds the report.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
if ROOT.name != "economic-data":
    candidate = ROOT / "experiments" / "economic-data"
    if candidate.exists():
        ROOT = candidate

sys.path.insert(0, str(ROOT))
from compute import build_facts
from fetch_world_bank import load_panel
from panels import INDICATORS, to_wide
from render_report import build_report

long = load_panel()
wide = to_wide(long)
focus = wide[wide["is_focus"]].copy()
countries = wide[~wide["is_aggregate"]].copy()
facts = build_facts(long)

print(
    f"{countries['iso'].nunique()} countries, "
    f"years {int(wide['year'].min())}–{int(wide['year'].max())}"
)
print(
    f"Ridge MAE {facts.forecast.test_mae} vs naive {facts.forecast.naive_mae} "
    f"(beats naive: {facts.forecast.beats_naive})"
)
display(focus[INDICATORS].describe().round(2))
display(pd.DataFrame([row.model_dump() for row in facts.focus])[["iso", "country", "latest_year", "flags"]])

In [ ]:
print("Pooled correlations (six-country panel):")
display(focus[INDICATORS].corr().round(3))

print("\nWithin-country correlations:")
display(
    focus.groupby("country")[INDICATORS]
    .corr()
    .unstack()
    .loc[:, [("inflation", "unemployment"), ("gdp_growth", "inflation")]]
    .round(3)
)

In [ ]:
print(facts.regression.formula, f"n={facts.regression.n}", f"R²={facts.regression.r_squared}")
display(pd.DataFrame([t.model_dump() for t in facts.regression.terms]))
print("\nForecasts:")
display(pd.DataFrame([p.model_dump() for p in facts.forecast.predictions]))

In [ ]:
from brief import generate_briefing

briefing = generate_briefing(facts)
report_path = build_report(long, facts=facts, briefing=briefing)
status = briefing.skipped_reason or briefing.model
display(HTML(
    f'<p>Wrote <a href="{report_path.name}" target="_blank">{report_path.name}</a> '
    f"({status}). Open it for charts, the forecast table, and the briefing.</p>"
))